# Proyecto 6 - Sistemas Recomendadores

**Alumno:** Jose Luis Cruz Gomez

## Objetivos

- Implementar un recomendador simple.
- Implementar un recomendador basado en contenido.
- Mejorar el recomendador utilizando metadatos.

In [1]:
# Importar librerías

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics.pairwise import cosine_similarity

from ast import literal_eval

In [2]:
# Cargar el dataset principal

df = pd.read_csv("./data/movies_metadata.csv", low_memory=False)

# Mostrar las primeras filas
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [3]:
# Ver tamaño del dataset

print("Filas y columnas:", df.shape)

Filas y columnas: (45466, 24)


In [ ]:
C = df["vote_average"].mean()

print("Calificación promedio:", C)

Calificación promedio: 5.618207215134184


In [6]:
m = df["vote_count"].quantile(0.90)

print("Mínimo de votos requerido:", m)

Mínimo de votos requerido: 160.0


In [7]:
q_movies = df.copy().loc[df["vote_count"] >= m]

print(q_movies.shape)

(4555, 24)


In [8]:
# Función para calcular la media ponderada

def weighted_rating(x, m=m, C=C):
    v = x["vote_count"]
    R = x["vote_average"]

    return (v / (v + m) * R) + (m / (m + v) * C)

In [9]:
# Calcular la puntuación ponderada

q_movies["score"] = q_movies.apply(weighted_rating, axis=1)

q_movies.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,score
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,7.640253
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,6.820293
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,5.660700
5,False,NaN,60000000,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",NaN,949,tt0113277,en,Heat,"Obsessive master thief, Neil McCauley leads a ...",...,187436818.0,170.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,A Los Angeles Crime Saga,Heat,False,7.7,1886.0,7.537201
8,False,NaN,35000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",NaN,9091,tt0114576,en,Sudden Death,International action superstar Jean Claude Van...,...,64350171.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Terror goes into overtime.,Sudden Death,False,5.5,174.0,5.556626


In [10]:
# Ordenar las películas por score

q_movies = q_movies.sort_values("score", ascending=False)

q_movies.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,score
314,False,NaN,25000000,"[{'id': 18, 'name': 'Drama'}, {'id': 80, 'name...",NaN,278,tt0111161,en,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,...,2.834147e+07,142.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Fear can hold you prisoner. Hope can set you f...,The Shawshank Redemption,False,8.5,8358.0,8.445869
834,False,"{'id': 230, 'name': 'The Godfather Collection'...",6000000,"[{'id': 18, 'name': 'Drama'}, {'id': 80, 'name...",http://www.thegodfather.com/,238,tt0068646,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",...,2.450664e+08,175.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,An offer you can't refuse.,The Godfather,False,8.5,6024.0,8.425439
10309,False,NaN,13200000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,19404,tt0112870,hi,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...",...,1.000000e+08,190.0,"[{'iso_639_1': 'hi', 'name': 'हिन्दी'}]",Released,Come... Fall In Love,Dilwale Dulhania Le Jayenge,False,9.1,661.0,8.421453
12481,False,"{'id': 263, 'name': 'The Dark Knight Collectio...",185000000,"[{'id': 18, 'name': 'Drama'}, {'id': 28, 'name...",http://thedarkknight.warnerbros.com/dvdsite/,155,tt0468569,en,The Dark Knight,Batman raises the stakes in his war on crime. ...,...,1.004558e+09,152.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Why So Serious?,The Dark Knight,False,8.3,12269.0,8.265477
2843,False,NaN,63000000,"[{'id': 18, 'name': 'Drama'}]",http://www.foxmovies.com/movies/fight-club,550,tt0137523,en,Fight Club,A ticking-time-bomb insomniac and a slippery s...,...,1.008538e+08,139.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Mischief. Mayhem. Soap.,Fight Club,False,8.3,9678.0,8.256385


In [11]:
# Mostrar las 15 mejores películas

q_movies[["title", "vote_count", "vote_average", "score"]].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


## Conclusión del recomendador simple

Se implementó un sistema de recomendación simple utilizando la fórmula de media ponderada de IMDb. Este método considera tanto la calificación promedio de cada película como la cantidad de votos recibidos, lo que permite obtener un ranking más confiable que usar únicamente la calificación promedio.

In [12]:
# Mostrar algunas descripciones de películas

df["overview"].head()

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: str

In [13]:
# Reemplazar valores vacíos

df["overview"] = df["overview"].fillna("")

# Crear el vectorizador TF-IDF

tfidf = TfidfVectorizer(stop_words="english")

# Construir la matriz TF-IDF

tfidf_matrix = tfidf.fit_transform(df["overview"])

# Mostrar dimensiones

tfidf_matrix.shape

(45466, 75827)

In [14]:
# Mostrar algunas palabras del vocabulario

tfidf.get_feature_names_out()[5000:5010]

array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

In [15]:
# Calcular la similitud entre todas las películas

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

cosine_sim.shape

(45466, 45466)

In [16]:
# Crear un índice con los títulos

indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

indices.head(10)

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
Heat                           5
Sabrina                        6
Tom and Huck                   7
Sudden Death                   8
GoldenEye                      9
dtype: int64

In [ ]:
# Función para recomendar películas similares

def get_recommendations(title, cosine_sim=cosine_sim):

    # Obtener el índice de la película
    idx = indices[title]

    # Calcular similitudes
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Ordenar de mayor a menor
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Tomar las 10 mejores 
    sim_scores = sim_scores[1:11]

    # Obtener índices
    movie_indices = [i[0] for i in sim_scores]

    # Regresar títulos
    return df["title"].iloc[movie_indices]

In [18]:
get_recommendations("The Dark Knight Rises")

12481                                      The Dark Knight
150                                         Batman Forever
1328                                        Batman Returns
15511                           Batman: Under the Red Hood
585                                                 Batman
21194    Batman Unmasked: The Psychology of the Dark Kn...
9230                    Batman Beyond: Return of the Joker
18035                                     Batman: Year One
19792              Batman: The Dark Knight Returns, Part 1
3095                          Batman: Mask of the Phantasm
Name: title, dtype: str

In [19]:
get_recommendations("The Godfather")

1178               The Godfather: Part II
44030    The Godfather Trilogy: 1972-1990
1914              The Godfather: Part III
23126                          Blood Ties
11297                    Household Saints
34717                   Start Liquidation
10821                            Election
38030            A Mother Should Be Loved
17729                   Short Sharp Shock
26293                  Beck 28 - Familjen
Name: title, dtype: str

## Conclusión del recomendador basado en contenido

En esta parte se implementó un sistema de recomendación utilizando las descripciones de las películas. Se aplicó la técnica TF-IDF para convertir el texto en vectores numéricos y posteriormente se utilizó la similitud coseno para encontrar películas con descripciones parecidas.

In [20]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [21]:
# Cargar datasets adicionales

credits = pd.read_csv("./data/credits.csv")
keywords = pd.read_csv("./data/keywords.csv")

print("Credits:", credits.shape)
print("Keywords:", keywords.shape)

Credits: (45476, 3)
Keywords: (46419, 2)


In [22]:
# Eliminar algunos registros problemáticos

df = df.drop([19730, 29503, 35587, 35803])

# Convertir IDs a enteros

keywords["id"] = keywords["id"].astype(int)
credits["id"] = credits["id"].astype(int)
df["id"] = df["id"].astype(int)

# Unir los datasets

df = df.merge(credits, on="id")
df = df.merge(keywords, on="id")

df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,spoken_languages,status,tagline,title,video,vote_average,vote_count,cast,crew,keywords
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...","[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...","[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [23]:
# Convertir texto serializado a listas

features = ["cast", "crew", "keywords", "genres"]

for feature in features:
    df[feature] = df[feature].apply(literal_eval)

In [24]:
# Obtener director

def get_director(x):
    for i in x:
        if i["job"] == "Director":
            return i["name"]
    return np.nan

In [25]:
# Obtener máximo 3 elementos

def get_list(x):
    if isinstance(x, list):
        names = [i["name"] for i in x]

        if len(names) > 3:
            names = names[:3]

        return names

    return []

In [26]:
# Director

df["director"] = df["crew"].apply(get_director)

# Actores, palabras clave y géneros

features = ["cast", "keywords", "genres"]

for feature in features:
    df[feature] = df[feature].apply(get_list)

df[["title","cast","director","keywords","genres"]].head()

,title,cast,director,keywords,genres
0,Toy Story,"[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter,"[jealousy, toy, boy]","[Animation, Comedy, Family]"
1,Jumanji,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]",Joe Johnston,"[board game, disappearance, based on children'...","[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Walter Matthau, Jack Lemmon, Ann-Margret]",Howard Deutch,"[fishing, best friend, duringcreditsstinger]","[Romance, Comedy]"
3,Waiting to Exhale,"[Whitney Houston, Angela Bassett, Loretta Devine]",Forest Whitaker,"[based on novel, interracial relationship, sin...","[Comedy, Drama, Romance]"
4,Father of the Bride Part II,"[Steve Martin, Diane Keaton, Martin Short]",Charles Shyer,"[baby, midlife crisis, confidence]",[Comedy]


In [27]:
# Limpiar texto

def clean_data(x):

    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]

    else:
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))

        return ""

In [28]:
features = ["cast","keywords","director","genres"]

for feature in features:
    df[feature] = df[feature].apply(clean_data)

In [29]:
# Crear la sopa de metadatos

def create_soup(x):
    return (
        " ".join(x["keywords"]) + " " +
        " ".join(x["cast"]) + " " +
        x["director"] + " " +
        " ".join(x["genres"])
    )

df["soup"] = df.apply(create_soup, axis=1)

df[["title","soup"]].head()

,title,soup
0,Toy Story,jealousy toy boy tomhanks timallen donrickles ...
1,Jumanji,boardgame disappearance basedonchildren'sbook ...
2,Grumpier Old Men,fishing bestfriend duringcreditsstinger walter...
3,Waiting to Exhale,basedonnovel interracialrelationship singlemot...
4,Father of the Bride Part II,baby midlifecrisis confidence stevemartin dian...


In [30]:
# Crear matriz de conteo

count = CountVectorizer(stop_words="english")

count_matrix = count.fit_transform(df["soup"])

count_matrix.shape

(46626, 73880)

In [31]:
# Similitud coseno

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

In [32]:
# Reiniciar índice

df = df.reset_index()

indices = pd.Series(df.index, index=df["title"])

In [33]:
get_recommendations("The Dark Knight Rises", cosine_sim2)

12541      The Dark Knight
10170        Batman Begins
9271                Shiner
9834       Amongst Friends
7732              Mitchell
516      Romeo Is Bleeding
11411         The Prestige
24040            Quicksand
24984             Deadfall
41041                 Sara
Name: title, dtype: str

In [34]:
get_recommendations("The Godfather", cosine_sim2)

1926            The Godfather: Part III
1187             The Godfather: Part II
15534                   The Rain People
18866                         Last Exit
34458                              Rege
35772            Manuscripts Don't Burn
35773            Manuscripts Don't Burn
7961     The Night of the Following Day
18187                 The Son of No One
28637            In the Name of the Law
Name: title, dtype: str

## Conclusión del recomendador mejorado

En este sistema se utilizaron actores principales, director, géneros y palabras clave para generar recomendaciones más precisas. Al incluir más información de cada película, el sistema logra encontrar similitudes que no pueden detectarse utilizando únicamente la descripción de la trama.

# Conclusión general

En este proyecto se implementaron tres sistemas de recomendación. El primero utilizó una media ponderada basada en la cantidad de votos y la calificación promedio de IMDb. El segundo empleó procesamiento de lenguaje natural con TF-IDF para recomendar películas según la similitud de sus descripciones. Finalmente, el tercer sistema incorporó actores, director, géneros y palabras clave, logrando recomendaciones más completas y precisas. Este proyecto permitió comprender cómo funcionan los sistemas de recomendación y cómo pueden mejorarse utilizando diferentes tipos de información.